In [2]:
import os
import pandas as pd
import numpy as np
from transformers import AutoTokenizer

# Define paths as provided
BASE_DATA_DIR = "/groups/orentsur_group/work/omertole/mimic_data"
IMAGE_DIR = os.path.join(BASE_DATA_DIR, "official_data_iccv_final")
TRAIN_CSV_PATH = os.path.join(BASE_DATA_DIR, "mimic_cxr_processed_train.csv")
VAL_CSV_PATH = os.path.join(BASE_DATA_DIR, "mimic_cxr_processed_validate.csv")

# Load CLIP tokenizer to measure exact model tokens
TOKENIZER_NAME = "openai/clip-vit-base-patch32"
print(f"Loading tokenizer: {TOKENIZER_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

def analyze_token_lengths(df_path, split_name="Train"):
    print(f"\n===========================================")
    print(f"Analyzing {split_name} Split from: {os.path.basename(df_path)}")
    print(f"===========================================")
    
    # Load dataset
    df = pd.read_csv(df_path)
    print(f"Loaded {len(df):,} rows.")
    
    # Fill NaN values with empty strings safely
    df['findings_clean'] = df['findings_clean'].fillna("").astype(str)
    df['impression_clean'] = df['impression_clean'].fillna("").astype(str)
    
    # Helper function to compute exact token count using the HF tokenizer
    def get_token_count(text):
        if not text.strip():
            return 0
        return len(tokenizer.tokenize(text))

    # Calculate token counts for both columns
    print("Calculating token counts for 'findings_clean'...")
    df['findings_tokens'] = df['findings_clean'].apply(get_token_count)
    
    print("Calculating token counts for 'impression_clean'...")
    df['impression_tokens'] = df['impression_clean'].apply(get_token_count)
    
    # Print metrics for each column
    for col_name, token_col in [("Findings Clean", "findings_tokens"), ("Impression Clean", "impression_tokens")]:
        lengths = df[token_col].values
        
        # Calculate statistics
        mean_len = np.mean(lengths)
        median_len = np.median(lengths)
        max_len = np.max(lengths)
        
        # Calculate percentage of rows exceeding CLIP's 77 token limit
        exceeds_77 = np.sum(lengths > 77)
        pct_exceeds_77 = (exceeds_77 / len(lengths)) * 100
        
        print(f"\n[{col_name} Token Statistics]")
        print(f"-> Average (Mean) Tokens : {mean_len:.2f}")
        print(f"-> Median Tokens         : {median_len:.1f}")
        print(f"-> Maximum Tokens        : {max_len}")
        print(f"-> Exceeds CLIP limit (>77 tokens): {exceeds_77:,} rows ({pct_exceeds_77:.2f}%)")

if __name__ == "__main__":
    # Run the analysis for both train and validation splits
    analyze_token_lengths(TRAIN_CSV_PATH, split_name="Train")
    analyze_token_lengths(VAL_CSV_PATH, split_name="Validation")

Loading tokenizer: openai/clip-vit-base-patch32...

Analyzing Train Split from: mimic_cxr_processed_train.csv


KeyboardInterrupt: 